# 📊 TechQA Model Evaluation: Base Llama 3.2-3B vs. Fine-tuned (`AQUABOT/Llama-3.2-3B-TechQA`)

- **Môn học:** Statistical Learning (Học máy thống kê) — HCMUS
- **Mục tiêu:** Đánh giá độc lập năng lực sinh câu trả lời kỹ thuật (Standalone LLM Evaluation) trước khi áp dụng RAG.
- **Tập dữ liệu kiểm thử:** `dev_Q_A.json` (160 câu hỏi Unseen có câu trả lời từ chuyên gia IBM).
- **Mô hình đối đầu:**
  1. **Base Model:** `unsloth/Llama-3.2-3B-Instruct` (Meta gốc)
  2. **Fine-tuned Model:** `AQUABOT/Llama-3.2-3B-TechQA` (Đã huấn luyện 3 Epochs trên TechQA)
- **Hệ thống chỉ số:** Exact Match (EM), Token-level F1, ROUGE-1/2/L, BLEU-1/4, Perplexity.

## 1. Cài đặt Thư viện & Cấu hình Môi trường
Cài đặt các thư viện đo lường và hiển thị trực quan dữ liệu.

In [ ]:
%%capture
!pip install -q transformers accelerate bitsandbytes evaluate rouge-score nltk matplotlib seaborn pandas tqdm datasets

## 2. Tải & Chuẩn bị Tập Dữ liệu Đánh giá (`dev_Q_A.json`)
Đọc 160 câu hỏi có đáp án (`ANSWERABLE == 'Y'`) từ tập `dev_Q_A.json` làm Ground Truth.

In [ ]:
import json
import pandas as pd

# Đọc tập dev_Q_A.json
with open("dev_Q_A.json", "r", encoding="utf-8") as f:
    dev_raw = json.load(f)

# Lọc các câu có đáp án
eval_samples = []
for item in dev_raw:
    if item.get("ANSWERABLE") == "Y":
        q = item["QUESTION_TITLE"].strip()
        if item.get("QUESTION_TEXT", "").strip():
            q += "\n\n" + item["QUESTION_TEXT"].strip()
        eval_samples.append({
            "question_id": item.get("QUESTION_ID", ""),
            "question": q,
            "ground_truth": item["ANSWER"].strip(),
            "document": item.get("DOCUMENT", "")
        })

df_eval = pd.DataFrame(eval_samples)
print(f"✅ Đã nạp thành công {len(df_eval)} câu hỏi đánh giá độc lập (Unseen Dev Set).")
print("\n--- Mẫu câu hỏi đầu tiên (Item 0) ---")
print(f"❓ Question: {df_eval.iloc[0]['question'][:200]}...")
print(f"🎯 Ground Truth: {df_eval.iloc[0]['ground_truth'][:200]}...")

## 3. Khởi tạo & Tải 2 Mô hình Đối đầu
Nạp đồng thời **Base Model** và **Fine-tuned Model (`AQUABOT/Llama-3.2-3B-TechQA`)**.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

BASE_MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"
FINETUNED_MODEL_ID = "AQUABOT/Llama-3.2-3B-TechQA"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Thiết bị sử dụng: {device}")

# 1. Tải Base Model
print("\n⏳ Đang tải Base Model (Meta Llama 3.2-3B)... ")
tokenizer_base = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
model_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)

# 2. Tải Fine-tuned Model
print("\n⏳ Đang tải Fine-tuned Model (AQUABOT/Llama-3.2-3B-TechQA)... ")
tokenizer_ft = AutoTokenizer.from_pretrained(FINETUNED_MODEL_ID)
model_ft = AutoModelForCausalLM.from_pretrained(
    FINETUNED_MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)

print("\n✅ Đã tải xong cả 2 mô hình thành công!")

## 4. Chạy Batch Inference (Sinh câu trả lời trên 160 câu)
Áp dụng Chat Template chuẩn và sinh câu trả lời với cùng một tham số kiểm soát (`temperature=0.7`, `max_new_tokens=256`).

In [ ]:
from tqdm import tqdm

SYSTEM_PROMPT = (
    "You are a technical support assistant specialized in IBM products. "
    "Answer the user's technical question accurately and concisely "
    "based on your knowledge of IBM technotes and documentation."
)

def generate_answer(model, tokenizer, question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    input_len = inputs.shape[1]
    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Chạy thử nghiệm trên toàn bộ 160 mẫu (hoặc sample 50 mẫu nếu muốn test nhanh)
NUM_TEST_SAMPLES = len(df_eval)  # 160 câu
test_subset = df_eval.iloc[:NUM_TEST_SAMPLES]

preds_base = []
preds_ft = []

print(f"🚀 Đang chạy inference trên {NUM_TEST_SAMPLES} câu hỏi...")
for idx, row in tqdm(test_subset.iterrows(), total=NUM_TEST_SAMPLES, desc="Generating Answers"):
    q = row["question"]
    ans_base = generate_answer(model_base, tokenizer_base, q)
    ans_ft = generate_answer(model_ft, tokenizer_ft, q)
    preds_base.append(ans_base)
    preds_ft.append(ans_ft)

test_subset["pred_base"] = preds_base
test_subset["pred_finetuned"] = preds_ft

# Lưu kết quả ra file JSON
test_subset.to_json("evaluation_predictions.json", orient="records", indent=2, force_ascii=False)
print("\n✅ Đã hoàn thành sinh câu trả lời và lưu tại 'evaluation_predictions.json'.")

## 5. Tính toán Bộ Chỉ số Định lượng (Quantitative Metrics Suite)
Tính các chỉ số: **Exact Match (EM)**, **Token-level F1**, **ROUGE-1, ROUGE-2, ROUGE-L**, **BLEU-1, BLEU-4**.

In [ ]:
import re
import string
import nltk
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

nltk.download('punkt', quiet=True)

def normalize_text(s):
    """Chuẩn hóa văn bản: viết thường, bỏ dấu câu và khoảng trắng thừa."""
    s = s.lower()
    s = re.sub(f"[{re.escape(string.punctuation)}]", " ", s)
    return " ".join(s.split())

def compute_exact_match(pred, target):
    return 1.0 if normalize_text(pred) == normalize_text(target) else 0.0

def compute_token_f1(pred, target):
    pred_tokens = normalize_text(pred).split()
    target_tokens = normalize_text(target).split()
    if not pred_tokens or not target_tokens:
        return 1.0 if pred_tokens == target_tokens else 0.0
    common = Counter(pred_tokens) & Counter(target_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(target_tokens)
    return (2 * precision * recall) / (precision + recall)

# Khởi tạo ROUGE scorer
rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smooth_fn = SmoothingFunction().method1

def evaluate_predictions(predictions, references):
    em_scores = []
    f1_scores = []
    r1_scores, r2_scores, rl_scores = [], [], []
    b1_scores, b4_scores = [], []
    
    for pred, ref in zip(predictions, references):
        em_scores.append(compute_exact_match(pred, ref))
        f1_scores.append(compute_token_f1(pred, ref))
        
        # ROUGE
        r_score = rouge.score(ref, pred)
        r1_scores.append(r_score['rouge1'].fmeasure * 100)
        r2_scores.append(r_score['rouge2'].fmeasure * 100)
        rl_scores.append(r_score['rougeL'].fmeasure * 100)
        
        # BLEU
        ref_tok = [normalize_text(ref).split()]
        pred_tok = normalize_text(pred).split()
        b1 = sentence_bleu(ref_tok, pred_tok, weights=(1, 0, 0, 0), smoothing_function=smooth_fn) * 100
        b4 = sentence_bleu(ref_tok, pred_tok, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth_fn) * 100
        b1_scores.append(b1)
        b4_scores.append(b4)
        
    return {
        "Exact Match (%)": round(sum(em_scores) / len(em_scores) * 100, 2),
        "Token F1-Score (%)": round(sum(f1_scores) / len(f1_scores) * 100, 2),
        "ROUGE-1 (%)": round(sum(r1_scores) / len(r1_scores), 2),
        "ROUGE-2 (%)": round(sum(r2_scores) / len(r2_scores), 2),
        "ROUGE-L (%)": round(sum(rl_scores) / len(rl_scores), 2),
        "BLEU-1 (%)": round(sum(b1_scores) / len(b1_scores), 2),
        "BLEU-4 (%)": round(sum(b4_scores) / len(b4_scores), 2),
    }

metrics_base = evaluate_predictions(test_subset["pred_base"], test_subset["ground_truth"])
metrics_ft = evaluate_predictions(test_subset["pred_finetuned"], test_subset["ground_truth"])

df_comparison = pd.DataFrame([
    {"Model": "Base Model (Llama 3.2-3B Gốc)", **metrics_base},
    {"Model": "Fine-tuned Model (AQUABOT TechQA)", **metrics_ft},
])

print("\n=========================================================================")
print("📊 BẢNG TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ ĐỐI ĐẦU TRỰC TIẾP (160 CÂU DEV SET)")
print("=========================================================================")
display(df_comparison)

## 6. Trực quan hóa Kết quả bằng Biểu đồ (Visualizations for Report)
Vẽ biểu đồ cột so sánh trực tiếp điểm số giữa Base Model và Fine-tuned Model để đưa vào Báo cáo đồ án.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)

plot_metrics = ["Token F1-Score (%)", "ROUGE-1 (%)", "ROUGE-L (%)", "BLEU-1 (%)", "BLEU-4 (%)"]
base_vals = [metrics_base[m] for m in plot_metrics]
ft_vals = [metrics_ft[m] for m in plot_metrics]

x = list(range(len(plot_metrics)))
width = 0.35

plt.figure(figsize=(12, 6), dpi=150)
bars1 = plt.bar([p - width/2 for p in x], base_vals, width, label="Base Llama 3.2-3B", color="#94a3b8")
bars2 = plt.bar([p + width/2 for p in x], ft_vals, width, label="Fine-tuned (AQUABOT TechQA)", color="#3b82f6")

# Thêm nhãn số trên đầu cột
for bar in bars1:
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5, f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=10, color="#475569")
for bar in bars2:
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5, f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold", color="#1d4ed8")

plt.xlabel("Metrics", fontweight="bold", labelpad=10)
plt.ylabel("Score (%)", fontweight="bold", labelpad=10)
plt.title("So sánh Hiệu năng: Base Model vs. Fine-tuned Model trên TechQA (160 câu Unseen)", fontweight="bold", pad=15, fontsize=14)
plt.xticks(x, plot_metrics)
plt.ylim(0, max(max(base_vals), max(ft_vals)) + 10)
plt.legend(frameon=True)
plt.tight_layout()
plt.savefig("techqa_model_comparison_chart.png", dpi=300)
plt.show()

print("💾 Biểu đồ đã được lưu thành file 'techqa_model_comparison_chart.png'!")

## 7. Trích xuất Các Case Study Định tính (Qualitative Side-by-Side Cases)
In ra 3 ca so sánh điển hình về phong cách trả lời và độ chính xác kỹ thuật để đưa vào bảng minh họa của Báo cáo.

In [ ]:
print("=========================================================================")
print("🔍 CÁC VÍ DỤ ĐỐI CHIẾU ĐỊNH TÍNH (QUALITATIVE COMPARISON CASE STUDIES)")
print("=========================================================================\n")

for i in range(min(3, len(test_subset))):
    row = test_subset.iloc[i]
    print(f"📌 [CASE #{i+1}] Question ID: {row['question_id']}")
    print(f"❓ CÂU HỎI:\n{row['question'][:300]}...\n")
    print(f"🎯 ĐÁP ÁN CHUẨN (GROUND TRUTH):\n{row['ground_truth']}\n")
    print(f"❌ BASE MODEL (Llama 3.2 Gốc):\n{row['pred_base'][:400]}...\n")
    print(f"✅ FINE-TUNED MODEL (AQUABOT TechQA):\n{row['pred_finetuned'][:400]}...\n")
    print("-" * 80 + "\n")